# Metaprogramación y Macros en Elixir

Elixir es un lenguaje construido sobre macros. De hecho, gran parte de su sintaxis (`if`, `def`, `defmodule`) son macros. La metaprogramación en Elixir te permite extender el lenguaje para adaptarlo a tus necesidades, creando DSLs (Domain Specific Languages) potentes.

## 1. El Árbol de Sintaxis Abstracta (AST)

Casi todo en Elixir se representa internamente como una estructura de datos llamada AST. Puedes inspeccionar esta estructura usando la macro `quote`.

In [ ]:
# quote convierte código en su representación AST
ast = quote do: 1 + 2
IO.inspect(ast)
# Resultado: {:+, [context: Elixir, import: Kernel], [1, 2]}

# Los tres elementos del AST son: {función/átomo, metadatos, argumentos}

## 2. Unquote

Dentro de un bloque `quote`, puedes usar `unquote` para inyectar valores externos dentro del código que se está generando.

In [ ]:
numero = 42
ast = quote do
  1 + unquote(numero)
end

IO.inspect(ast)
Code.eval_quoted(ast) # Evalúa el código

## 3. Definición de Macros

Las macros se definen con `defmacro`. Una macro recibe código (en forma de AST) y retorna código (también en AST) que se inyecta en el lugar de la llamada durante la compilación.

In [ ]:
defmodule MiLenguaje do
  defmacro decir_si(condicion, do: bloque) do
    quote do
      if unquote(condicion) do
        unquote(bloque)
      end
    end
  end
end

require MiLenguaje
MiLenguaje.decir_si 2 > 1 do
  IO.puts "¡Es mayor!"
end

## 4. Higiene de Macros

Elixir implementa macros higiénicas, lo que significa que las variables definidas dentro de una macro no colisionan con las del ámbito donde se invoca la macro.

In [ ]:
defmodule Higiene do
  defmacro sin_colision do
    quote do: x = 1
  end
end

x = 10
require Higiene
Higiene.sin_colision()
IO.puts x # Seguirá siendo 10

# Si realmente quieres afectar el ámbito externo, usa var!():
# quote do: var!(x) = 1

## 5. El uso de `__using__` e Inyección de Código

Este es un patrón muy común en librerías como Phoenix para inyectar código común en varios módulos.

In [ ]:
defmodule MiLibreria do
  defmacro __using__(_opts) do
    quote do
      def hola_desde_lib, do: IO.puts "Hola desde la gema"
    end
  end
end

defmodule MiModulo do
  use MiLibreria
end

MiModulo.hola_desde_lib()

## 6. Mejores Prácticas

1. **No uses macros si puedes usar funciones**: Las macros son más difíciles de leer y depurar.
2. **Usa `quote` para generar código**: Evita construir ASTs manualmente si es posible.
3. **Documenta tus macros**: Al ocultar lógica compleja, es vital que los usuarios sepan qué está pasando.
4. **Usa `Macro.expand`**: Para ver qué código está generando realmente tu macro durante el desarrollo.